# Notebook 15 — Agent Economics: Tokens → Work → Quality → Outcome → Value

This is the capstone notebook for the `deep-agents-on-foundry` learning journey.

The central question is not **how many tokens did the agent use?** but **what valuable outcome did those tokens and agentic actions produce?**

```text
TOKENS → AGENTIC WORK → INFORMATION GAIN → QUALITY GAIN → TASK OUTCOME → BUSINESS VALUE
```

By the end you should be able to measure inference cost, agentic work, information gain, quality, task success, human effort, failure cost, and business value; compare architectures on a value frontier; and reason about adaptive spend.

## 15.1 — Why token cost alone is misleading

Lower token cost does not automatically mean better economics, and higher quality does not automatically mean better economics. Economics is **value produced relative to total work and total cost consumed**.

Example: a 10K-token answer at 3/5 quality may be worse than a 20K-token answer at 5/5 if the task requires quality ≥ 4.5.

## 15.2 — The Token-to-Value ladder

```text
                  BUSINESS VALUE
                        ↑
                 Task Outcome
                        ↑
                  Quality Gain
                        ↑
                Information Gain
                        ↑
                  Agentic Work
                        ↑
               Inference / Tokens
```

| Level | Question |
|---|---|
| Tokens | What inference did we consume? |
| Agentic work | What did the system actually do? |
| Information gain | What useful new evidence did that work uncover? |
| Quality | Did the output become better? |
| Outcome | Did the task actually succeed? |
| Value | What was successful completion worth? |

## 15.3 — Layer 1: Inference economics

Track input/output/cached tokens, model calls, latency, search/tool calls, retrieval, embeddings, storage, and infrastructure.

```text
Total Agent Cost = Model Cost + Tool Cost + Retrieval Cost + Infrastructure Cost
```

This is necessary, but it is only the bottom rung.

In [ ]:
from dataclasses import dataclass, asdict
from typing import Optional

@dataclass
class RunEconomics:
    task_id: str
    architecture: str
    complexity: str
    input_tokens: int = 0
    output_tokens: int = 0
    cached_tokens: int = 0
    model_calls: int = 0
    web_searches: int = 0
    tool_calls: int = 0
    subagent_calls: int = 0
    memory_reads: int = 0
    skill_loads: int = 0
    summarizations: int = 0
    retries: int = 0
    latency_seconds: float = 0.0
    quality_score: Optional[float] = None
    citation_score: Optional[float] = None
    task_success: Optional[bool] = None
    estimated_agent_cost: Optional[float] = None
    human_review_minutes: Optional[float] = None
    human_rework_minutes: Optional[float] = None
    failure_cost: Optional[float] = None
    business_value_if_success: Optional[float] = None

## 15.4 — Layer 2: Agentic work

Tokens tell you how much computation happened, not why. Measure model calls, searches, tool calls, files read, memories retrieved, skills loaded, subagents invoked, summarizations, and retries.

In [ ]:
def agentic_work_summary(run: RunEconomics) -> dict:
    return {
        "model_calls": run.model_calls,
        "web_searches": run.web_searches,
        "tool_calls": run.tool_calls,
        "subagent_calls": run.subagent_calls,
        "memory_reads": run.memory_reads,
        "skill_loads": run.skill_loads,
        "summarizations": run.summarizations,
        "retries": run.retries,
    }

## 15.5 — Activity is not progress

Eight searches can contain only three useful discoveries. Agent activity is not the same as progress.

```text
activity ≠ progress
```

The goal is useful work, not maximal agentic activity.

## 15.6 — Layer 3: Information gain

Classify each search/tool/subagent result as:

```text
novel + useful
novel but irrelevant
duplicate
incorrect / noisy
```

This is a practical way to measure whether work produced new decision-useful evidence.

In [ ]:
from enum import Enum

class EvidenceContribution(str, Enum):
    NOVEL_USEFUL = "novel_useful"
    NOVEL_IRRELEVANT = "novel_irrelevant"
    DUPLICATE = "duplicate"
    NOISY = "noisy"

def information_gain_efficiency(labels):
    if not labels:
        return 0.0
    useful = sum(1 for x in labels if x == EvidenceContribution.NOVEL_USEFUL)
    return useful / len(labels)

## 15.7 — Information-gain efficiency

Agent A: 3 searches → 3 useful discoveries = 1.0 efficiency. Agent B: 10 searches → 4 useful discoveries = 0.4 efficiency.

This begins to expose **agentic waste**.

## 15.8 — Layer 4: Quality gain

Measure correctness, completeness, source quality, citation quality, architecture clarity, trade-off depth, and decision usefulness.

Keep separate deltas such as `ΔQuality`, `ΔTokens`, `ΔLatency`, and `ΔTool Cost`.

## 15.9 — Do not collapse economics into one magical score too early

A simple “quality gain per token” ratio can be misleading. If a cheaper system still fails the task threshold, its better ratio may not matter.

First measure quality, cost, latency, and tool usage separately. Then connect them to task success.

## 15.10 — Layer 5: Task outcome

Quality is a proxy. The user ultimately wants a successful task.

For a research agent, success might require: quality threshold met, required dimensions covered, citations grounded, no critical factual error, and little/no human rework.

In [ ]:
def task_success_from_quality(quality_score: float, threshold: float = 4.5, critical_error: bool = False) -> bool:
    return quality_score >= threshold and not critical_error

## 15.11 — Cost per successful task

If Baseline costs $0.08 but fails and Deep Agent costs $0.21 and succeeds, the more expensive run is economically superior.

```text
Cost per Successful Task = Total Cost / Successful Tasks
```

In [ ]:
def cost_per_success(total_cost: float, successful_tasks: int):
    return float("inf") if successful_tasks == 0 else total_cost / successful_tasks

## 15.12 — Layer 6: Business value

Translate successful completion into value: analyst time saved, support escalation avoided, developer time saved, proposal speed, fewer manual handoffs, etc.

```text
Expected Business Value = P(success) × Value(success)
Net Value = Expected Business Value − Agent Cost
```

Later we add human and failure costs.

## 15.13 — The ROI ladder

```text
TOKEN ECONOMICS
How much inference did we buy?
        ↓
AGENT ECONOMICS
What useful work did those tokens fund?
        ↓
TASK ECONOMICS
Did that work solve the task?
        ↓
BUSINESS ECONOMICS
What was successful task completion worth?
```

## 15.14 — Agentic waste taxonomy

1. repeated context
2. redundant searches
3. unnecessary tool calls
4. unnecessary subagents
5. failed retries
6. wrong tool selection
7. irrelevant memory retrieval
8. irrelevant skill loading
9. excessive summarization
10. re-research caused by lost context

Optimize **waste**, not blindly total tokens.

In [ ]:
AGENTIC_WASTE_CATEGORIES = [
    "repeated_context", "redundant_search", "unnecessary_tool_call",
    "unnecessary_subagent", "failed_retry", "wrong_tool",
    "irrelevant_memory", "irrelevant_skill", "excessive_summarization",
    "re_research_from_lost_context",
]

## 15.15 — Diminishing marginal returns

Example quality curve:

```text
0 searches → 2.5
1 search   → 4.0
2 searches → 4.6
3 searches → 4.8
4 searches → 4.82
5 searches → 4.83
```

Early work creates large gains; later work barely helps. The ideal stopping point is near the elbow.

In [ ]:
searches = [0,1,2,3,4,5]
quality = [2.5,4.0,4.6,4.8,4.82,4.83]
marginal_quality_gain = [None] + [quality[i]-quality[i-1] for i in range(1,len(quality))]
list(zip(searches, quality, marginal_quality_gain))

## 15.16 — Marginal agent economics

```text
Marginal Value(n) = Value after step n − Value after step n−1
Marginal Cost(n)  = Cost after step n − Cost after step n−1
```

Continue extra work while marginal value is greater than marginal cost. This is the economics interpretation of adaptive orchestration.

## 15.17 — Complexity routing

A likely pattern:

```text
simple  → model + search
medium  → Deep Agent
complex → Deep Agent + Skills/Subagents
```

This is **adaptive spend**: spend more agentic work only when the task justifies it.

## 15.18 — Agentic budgets

Budgets can constrain model calls, searches, subagents, input tokens, and latency based on task complexity/value.

In [ ]:
AGENTIC_BUDGETS = {
    "simple":  {"max_model_calls":3,  "max_searches":1, "max_subagents":0, "max_input_tokens":15000, "max_latency_seconds":20},
    "medium":  {"max_model_calls":6,  "max_searches":3, "max_subagents":1, "max_input_tokens":35000, "max_latency_seconds":60},
    "complex": {"max_model_calls":10, "max_searches":6, "max_subagents":2, "max_input_tokens":70000, "max_latency_seconds":120},
}

## 15.19 — The Value Frontier

Compare architectures such as model-only, model+search, Deep Agent, Deep Agent+Memory, +Skills, +Subagents.

Plot quality vs cost. Architectures that are both more expensive and lower quality are dominated. The remaining architectures form the **Pareto/value frontier**.

In [ ]:
def pareto_frontier(points):
    frontier=[]
    for p in points:
        dominated=False
        for q in points:
            if q is p: continue
            no_worse = q["cost"] <= p["cost"] and q["quality"] >= p["quality"]
            strictly_better = q["cost"] < p["cost"] or q["quality"] > p["quality"]
            if no_worse and strictly_better:
                dominated=True; break
        if not dominated: frontier.append(p)
    return frontier

## 15.20 — Economics of Memory

Memory can reduce repeated explanation/research/context, but adds retrieval/embedding/context cost and stale-memory risk. Measure whether it reduces total work and improves personalization enough to justify itself.

## 15.21 — Economics of Skills

Skills can replace large always-loaded procedural prompts with metadata + selective procedure loading. Measure routing accuracy, quality gain, token delta, and latency delta.

## 15.22 — Economics of Middleware

Summarization can reduce repeated context but costs a summarization call and may lose information. Evaluate:

```text
context saved − summary cost − quality loss
```

## 15.23 — Economics of Subagents

Subagents may improve specialization and parent-context efficiency while increasing total calls/tokens/coordination. They may improve **context efficiency** while worsening **compute efficiency**.

## 15.24 — Canonical experiment record

Use one row per run so architecture/task/repetition comparisons are consistent.

In [ ]:
import pandas as pd
sample_runs = [
    RunEconomics(task_id="simple-1",architecture="baseline_search",complexity="simple",input_tokens=9000,output_tokens=1200,model_calls=2,web_searches=1,latency_seconds=12,quality_score=4.0,task_success=False,estimated_agent_cost=0.08),
    RunEconomics(task_id="simple-1",architecture="deep_agent",complexity="simple",input_tokens=11000,output_tokens=1400,model_calls=3,web_searches=1,tool_calls=1,latency_seconds=10,quality_score=5.0,task_success=True,estimated_agent_cost=0.11),
]
sample_df = pd.DataFrame([asdict(x) for x in sample_runs])
sample_df

## 15.25 — Human effort often dominates token cost

An agent that costs $0.40 but needs 1 minute of editing can be far cheaper than a $0.05 agent that needs 15 minutes of editing.

```text
Effective Task Cost = Agent Cost + Human Review + Human Rework + Expected Failure Cost
```

In [ ]:
def human_cost(minutes: float, hourly_rate: float) -> float:
    return minutes/60 * hourly_rate

def effective_task_cost(agent_cost: float, review_minutes: float=0, rework_minutes: float=0, hourly_rate: float=100, failure_probability: float=0, failure_cost: float=0):
    return agent_cost + human_cost(review_minutes+rework_minutes, hourly_rate) + failure_probability*failure_cost

## 15.26 — Reliability belongs in economics

A 5× more expensive model can be economically cheaper if it reduces expensive failures. Always include expected failure cost for high-stakes workflows.

In [ ]:
cheap = effective_task_cost(0.10, failure_probability=0.10, failure_cost=100)
expensive = effective_task_cost(0.50, failure_probability=0.01, failure_cost=100)
print("cheap:", cheap, "expensive:", expensive)

## 15.27 — Expected Net Value

```text
Expected Net Value = P(success)×Value(success) − P(failure)×Cost(failure) − Agent Cost − Human Cost
```

In [ ]:
def expected_net_value(p_success, value_if_success, cost_if_failure, agent_cost, human_cost_total=0):
    return p_success*value_if_success - (1-p_success)*cost_if_failure - agent_cost - human_cost_total

## 15.28 — Token Margin vs Value Margin

```text
Token Margin = Revenue − Inference Cost
Value Margin = Value Delivered − Total Cost to Deliver
```

Total delivery cost includes inference, tools, infrastructure, human review/rework, and failure cost.

## 15.29 — Customer-facing efficiency ladder

1. Token Efficiency — are we reducing unnecessary inference?
2. Work Efficiency — is agentic work productive?
3. Quality Efficiency — does extra work improve output quality?
4. Outcome Efficiency — does higher quality improve task success?
5. Value Efficiency — is success worth more than total delivery cost?

## 15.30 — Optimization order

1. Make the task succeed reliably
2. Remove obviously wasteful work
3. Route complexity appropriately
4. Reduce unnecessary context
5. Optimize model/tool selection
6. Optimize infrastructure

Do not optimize yourself into a **cheap failure**.

## 15.31 — Capstone experiment design

Compare:

A. Baseline model + Web Search
B. Deep Agent
C. Deep Agent + Skills
D. Deep Agent + Skills + Subagents

Across simple / medium / complex tasks, 3 repetitions each = **36 runs**.

In [ ]:
ARCHITECTURES=["baseline_search","deep_agent","deep_agent_skills","deep_agent_skills_subagents"]
COMPLEXITIES=["simple","medium","complex"]
NUM_REPETITIONS=3
planned_runs=[{"architecture":a,"complexity":c,"rep":r} for a in ARCHITECTURES for c in COMPLEXITIES for r in range(1,NUM_REPETITIONS+1)]
len(planned_runs)

## 15.32 — Suggested tasks

**Simple:** What is Microsoft Foundry Hosted Agents?

**Medium:** Explain Foundry Hosted Agents architecture, state/session model, and identity boundaries.

**Complex:** Compare Foundry Hosted Agents with a custom AKS-hosted LangGraph agent for enterprise workloads. Cover architecture, identity, state ownership, scaling, observability, reliability, lock-in, and economics.

Complexity should come from the task, not artificial prompt length.

## 15.33 — Record for every run

At minimum capture quality, task success, input/output tokens, latency, model calls, searches, tool calls, subagent calls, skill loads, summarizations, retries, and estimated cost. Later add human review/rework, failure cost, and business value.

In [ ]:
def add_derived_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out=df.copy()
    out["total_tokens"]=out["input_tokens"]+out["output_tokens"]
    out["successful_task_int"]=out["task_success"].fillna(False).astype(int)
    return out

def summarize_experiment(df: pd.DataFrame) -> pd.DataFrame:
    temp=add_derived_metrics(df)
    return (temp.groupby(["complexity","architecture"],dropna=False)
        .agg(runs=("task_id","count"),avg_quality=("quality_score","mean"),success_rate=("successful_task_int","mean"),avg_total_tokens=("total_tokens","mean"),avg_latency=("latency_seconds","mean"),avg_searches=("web_searches","mean"),avg_subagents=("subagent_calls","mean"),avg_cost=("estimated_agent_cost","mean"))
        .reset_index())

## 15.34 — State hypotheses before running

A plausible hypothesis:

```text
Simple  → baseline wins economically
Medium  → Deep Agent / Skills may win
Complex → Skills + selective Subagents may win
```

But the experiment must be allowed to prove this wrong.

## 15.35 — Visualize the quality–cost frontier

Use cost or total tokens on x-axis and quality/success on y-axis. Plot separate frontiers by task complexity.

In [ ]:
import matplotlib.pyplot as plt

def plot_quality_cost_frontier(summary_df, complexity: str):
    subset=summary_df[summary_df["complexity"]==complexity].copy()
    fig,ax=plt.subplots(figsize=(8,5))
    for _,row in subset.iterrows():
        ax.scatter(row["avg_cost"],row["avg_quality"])
        ax.annotate(row["architecture"],(row["avg_cost"],row["avg_quality"]),xytext=(5,5),textcoords="offset points")
    ax.set_xlabel("Average estimated cost")
    ax.set_ylabel("Average quality")
    ax.set_title(f"Quality–Cost Frontier: {complexity}")
    plt.show()

## 15.36 — Adaptive router as the likely end-state

```text
                     User Task
                         ↓
                Complexity Router
              /          |          \
          Simple       Medium       Complex
            ↓             ↓             ↓
     Model + Search   Deep Agent   Deep Agent + Skills + Subagents
```

The design principle is **minimum sufficient agentic work**.

In [ ]:
def choose_architecture(complexity: str) -> str:
    return {"simple":"baseline_search","medium":"deep_agent","complex":"deep_agent_skills_subagents"}[complexity]

## 15.37 — Core optimization objective

Conceptually maximize:

```text
Business Value / Total Agentic Work
```

subject to:

```text
quality ≥ required threshold
reliability ≥ required threshold
latency ≤ acceptable threshold
```

This is fundamentally different from minimizing tokens.

## 15.38 — Agent Economics scorecard

| Dimension | Question |
|---|---|
| Token efficiency | How much inference did we consume? |
| Work efficiency | How much agentic work was useful? |
| Information efficiency | How much novel useful evidence was produced? |
| Quality | Did output improve? |
| Success rate | Did it cross the required threshold? |
| Effective cost | Agent + human + failure cost |
| Value | What was successful completion worth? |
| Net value | Value minus total delivery cost |

## 15.39 — Practical implementation order

**Phase 1 — Engineering economics:** tokens, latency, calls, searches, tools, subagents, quality, success.

**Phase 2 — Human economics:** review time, rework time, acceptance rate.

**Phase 3 — Business economics:** task value, failure cost, expected net value.

Do not pretend all business value is measurable on day one.

## 15.40 — Connect back to the earlier Deep Agent experiment

The earlier baseline-vs-Deep experiment already hinted at the central pattern: simple tasks can be over-agented, while harder tasks may justify deeper scaffolding.

This notebook turns that observation into:

```text
task complexity → right amount of agentic work → quality threshold → cost per successful task → business value
```

## 15.41 — Decision checklist

Before adding more agent complexity, ask:

1. What failure are we fixing?
2. What extra work will the new architecture perform?
3. Will that work produce useful new information?
4. What quality dimension should improve?
5. Does the improvement cross a real success threshold?
6. What extra tokens/calls/searches/subagents will it consume?
7. Does it reduce human rework?
8. What is failure worth?
9. What is successful completion worth?
10. Is the architecture on the value frontier for this task class?

# Notebook 15 — Key takeaways

1. Token economics is only the bottom layer.
2. Agentic work must create useful information.
3. Activity is not progress.
4. Quality matters when it changes task outcomes.
5. Cost per successful task is more useful than raw cost per run.
6. Human review/rework can dominate inference cost.
7. Reliability and failure cost belong in economics.
8. Memory, Skills, Middleware, and Subagents must each earn their complexity.
9. Marginal returns matter; more agentic work eventually stops paying off.
10. Different task complexities may justify different architectures.
11. The value frontier is more useful than one universal architecture.
12. The likely end state is adaptive orchestration: **minimum sufficient agentic work**.

Final framework:

```text
TOKENS → WORK → INFORMATION → QUALITY → OUTCOME → VALUE
```

> **Reduce wasted work, not useful intelligence.**

## Suggested next step

Run the 36-run capstone experiment and answer:
- which architecture wins by complexity?
- which architectures are dominated?
- where do marginal returns flatten?
- what should the adaptive router choose?
- which extra features actually earn their cost?